# 02. 시계열 전처리와 데이터 분할

용해탱크 KPI MLOps v1 — Notebook 기반 모델 개발

## 전처리 원칙

- 입력 특성: `MELT_TEMP`, `MOTORSPEED`, `MELT_WEIGHT`
- 한 timestamp의 10개 행을 하나의 LSTM 시퀀스로 사용
- 현재 분 `X[t]`로 다음 분 `y[t+1]` 예측
- 다음 분의 NG가 5개 이상이면 NG=1
- 시간순으로 train/validation/test-NG/test-normal 분할
- scaler는 train에만 `fit`

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / "data/raw/melting_tank.csv"
OUTPUT_DIR = PROJECT_ROOT / "data/processed"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = ["MELT_TEMP", "MOTORSPEED", "MELT_WEIGHT"]
SEQUENCE_LENGTH = 10
QUICK_RUN = False

In [2]:
df = pd.read_csv(DATA_PATH, usecols=["STD_DT", *FEATURES, "TAG"])
df["STD_DT"] = pd.to_datetime(df["STD_DT"], errors="raise")
df["target_ng"] = (df["TAG"] == "NG").astype("int8")
df = df.sort_values("STD_DT", kind="stable").reset_index(drop=True)

assert not df[[*FEATURES, "target_ng"]].isna().any().any()
print("rows:", len(df), "minutes:", df["STD_DT"].nunique())

rows: 835200 minutes: 83520


In [3]:
sequences, minute_targets, timestamps = [], [], []

for timestamp, group in df.groupby("STD_DT", sort=True):
    if len(group) != SEQUENCE_LENGTH:
        continue
    sequences.append(group[FEATURES].to_numpy(dtype="float32"))
    minute_targets.append(int(group["target_ng"].sum() >= 5))
    timestamps.append(np.datetime64(timestamp))

X_current = np.asarray(sequences, dtype="float32")
y_current = np.asarray(minute_targets, dtype="int8")
times_current = np.asarray(timestamps)

## 현재 분의 시퀀스로 다음 분의 상태를 예측
X, y, times = X_current[:-1], y_current[1:], times_current[:-1]
print("X:", X.shape, "y:", y.shape, "NG ratio:", y.mean())

X: (83519, 10, 3) y: (83519,) NG ratio: 0.18883128389947199


In [5]:
masks = {
    "train": times <= np.datetime64("2020-03-31T23:59:00"),
    "validation": (times > np.datetime64("2020-03-31T23:59:00")) & (times <= np.datetime64("2020-04-07T23:59:00")),
    "test_ng": (times > np.datetime64("2020-04-07T23:59:00")) & (times <= np.datetime64("2020-04-14T23:59:00")),
    "test_normal": times > np.datetime64("2020-04-14T23:59:00"),
}

if QUICK_RUN:
    ## 각 시간 구간 전체에서 고르게 추출하여 클래스와 분포 변화를 함께 확인한다.
    quick_masks = {}
    for name, mask in masks.items():
        indices = np.flatnonzero(mask)
        selected = indices[np.linspace(0, len(indices) - 1, min(4_000, len(indices)), dtype=int)]
        quick_mask = np.zeros(len(X), dtype=bool)
        quick_mask[selected] = True
        quick_masks[name] = quick_mask
    masks = quick_masks

summary = []

for name, mask in masks.items():
    assert mask.any(), f"{name} 분할이 비어 있습니다."
    summary.append({"split": name, "samples": int(mask.sum()), "ng": int(y[mask].sum()), "ng_ratio": float(y[mask].mean())})
    
pd.DataFrame(summary)

,split,samples,ng,ng_ratio
0,train,40320,12549,0.311235
1,validation,10080,2907,0.288393
2,test_ng,10080,315,0.031250
3,test_normal,23039,0,0.000000


In [6]:
split_order = ["train", "validation", "test_ng", "test_normal"]
X_splits = {name: X[mask] for name, mask in masks.items()}
y_splits = {name: y[mask] for name, mask in masks.items()}

scaler = MinMaxScaler()
n_features = len(FEATURES)
scaler.fit(X_splits["train"].reshape(-1, n_features))

for name in split_order:
    values = X_splits[name]
    X_splits[name] = scaler.transform(values.reshape(-1, n_features)).reshape(values.shape).astype("float32")
    np.save(OUTPUT_DIR / f"X_{name}.npy", X_splits[name])
    np.save(OUTPUT_DIR / f"y_{name}.npy", y_splits[name])

joblib.dump(scaler, ARTIFACT_DIR / "scaler.joblib")

metadata = {"features": FEATURES, "sequence_length": SEQUENCE_LENGTH, "target_rule": "next-minute-majority", "quick_run": QUICK_RUN}

(ARTIFACT_DIR / "preprocessing.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

print("전처리 산출물 저장 완료")

전처리 산출물 저장 완료


## 확인 사항

최종 입력 모양은 `(표본 수, 10, 3)`입니다. 검증 및 테스트 데이터의 값이 0~1 범위를 벗어날 수 있는데, 이는 학습 데이터 범위 밖의 값이 들어왔다는 의미이며 오류가 아닙니다.